%md
# Petroleum Pipeline — Gold Core

Reusable Gold schemas, table setup, preparation functions,
audited loaders, and refresh orchestration.

Data loading occurs only when refresh_gold is called.

In [0]:
from datetime import date, datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType,
    TimestampType,
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

silver_table = "workspace.default.petroleum_silver"
execution_log_table = "workspace.default.petroleum_execution_logs"
gold_table_prefix = "workspace.default.petroleum_gold"

gold_tables = {
    "date": f"{gold_table_prefix}_dim_date",
    "product": f"{gold_table_prefix}_dim_product",
    "area": f"{gold_table_prefix}_dim_area",
    "process": f"{gold_table_prefix}_dim_process",
    "unit": f"{gold_table_prefix}_dim_unit",
    "fact": f"{gold_table_prefix}_fact_observation",
}

gold_schemas = {
    "date": StructType([
        StructField("date_key", IntegerType(), False),
        StructField("calendar_date", DateType(), False),
        StructField("year", IntegerType(), False),
        StructField("quarter", IntegerType(), False),
        StructField("month_number", IntegerType(), False),
        StructField("month_name", StringType(), False),
        StructField("year_month", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),
    "product": StructType([
        StructField("product_key", StringType(), False),
        StructField("product_code", StringType(), False),
        StructField("product_name", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),
    "area": StructType([
        StructField("area_key", StringType(), False),
        StructField("area_code", StringType(), False),
        StructField("area_name", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),
    "process": StructType([
        StructField("process_key", StringType(), False),
        StructField("process_code", StringType(), False),
        StructField("process_name", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),
    "unit": StructType([
        StructField("unit_key", StringType(), False),
        StructField("unit_code", StringType(), False),
        StructField("unit_description", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),
    "fact": StructType([
        StructField("observation_key", StringType(), False),
        StructField("date_key", IntegerType(), False),
        StructField("product_key", StringType(), False),
        StructField("area_key", StringType(), False),
        StructField("process_key", StringType(), False),
        StructField("unit_key", StringType(), False),
        StructField("series", StringType(), False),
        StructField("value", DoubleType(), True),
        StructField("record_hash", StringType(), False),
        StructField("silver_load_timestamp", TimestampType(), False),
        StructField("run_id", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),
}

for required_table in [silver_table, execution_log_table]:
    if not spark.catalog.tableExists(required_table):
        raise ValueError(f"Required table is missing: {required_table}")

# Preserve existing tables and validate their contracts.
for table_role, table_name in gold_tables.items():
    expected_schema = gold_schemas[table_role]

    column_definitions = ",\n".join(
        f"`{field.name}` {field.dataType.simpleString()}"
        + (" NOT NULL" if not field.nullable else "")
        for field in expected_schema.fields
    )

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            {column_definitions}
        )
        USING DELTA
    """)

    actual_schema = spark.table(table_name).schema

    expected_fields = [
        (field.name, field.dataType.simpleString(), field.nullable)
        for field in expected_schema.fields
    ]
    actual_fields = [
        (field.name, field.dataType.simpleString(), field.nullable)
        for field in actual_schema.fields
    ]

    if actual_fields != expected_fields:
        raise ValueError(f"Schema mismatch for {table_name}")

# Add optional Gold metrics without replacing existing audit records.
existing_log_columns = set(spark.table(execution_log_table).columns)

for column_name in ["rows_output", "rows_aggregated"]:
    if column_name not in existing_log_columns:
        spark.sql(f"""
            ALTER TABLE {execution_log_table}
            ADD COLUMNS ({column_name} BIGINT)
        """)

print("Gold configuration and table schemas ready.")

In [0]:
dimension_specs = {
    "product": {
        "source_code": "product",
        "source_name": "product_name",
        "target_code": "product_code",
        "target_name": "product_name",
        "key_column": "product_key",
    },
    "area": {
        "source_code": "duoarea",
        "source_name": "area_name",
        "target_code": "area_code",
        "target_name": "area_name",
        "key_column": "area_key",
    },
    "process": {
        "source_code": "process",
        "source_name": "process_name",
        "target_code": "process_code",
        "target_name": "process_name",
        "key_column": "process_key",
    },
}


def prepare_descriptive_dimension(silver_source_df, *, dimension_role):
    """Prepare unique dimension members with stable source-code keys."""
    if dimension_role not in dimension_specs:
        raise ValueError(f"Unsupported dimension: {dimension_role}")

    spec = dimension_specs[dimension_role]
    code_column = spec["target_code"]
    name_column = spec["target_name"]

    members_df = silver_source_df.select(
        F.col(spec["source_code"]).alias(code_column),
        F.col(spec["source_name"]).alias(name_column),
    ).distinct()

    invalid_members = members_df.filter(
        F.col(code_column).isNull()
        | (F.trim(F.col(code_column)) == "")
        | F.col(name_column).isNull()
        | (F.trim(F.col(name_column)) == "")
    )

    if invalid_members.limit(1).count() > 0:
        raise ValueError(
            f"{dimension_role}: missing source codes or descriptions."
        )

    conflicting_codes = (
        members_df.groupBy(code_column)
        .count()
        .filter(F.col("count") > 1)
    )

    if conflicting_codes.limit(1).count() > 0:
        raise ValueError(
            f"{dimension_role}: conflicting descriptions for a source code."
        )

    return (
        members_df
        .withColumn(
            spec["key_column"],
            F.sha2(
                F.to_json(
                    F.struct(
                        F.lit(dimension_role).alias("dimension"),
                        F.col(code_column).alias("code"),
                    )
                ),
                256,
            ),
        )
        .select(spec["key_column"], code_column, name_column)
    )


def prepare_date_dimension(silver_source_df):
    """Build a continuous calendar covering complete source years."""
    bounds = silver_source_df.agg(
        F.min("period").alias("minimum_period"),
        F.max("period").alias("maximum_period"),
    ).first()

    if bounds["minimum_period"] is None:
        raise ValueError("Cannot build a calendar from empty Silver data.")

    start_date = date(bounds["minimum_period"].year, 1, 1)
    end_date = date(bounds["maximum_period"].year, 12, 31)

    calendar_df = spark.range(1).select(
        F.explode(
            F.sequence(
                F.lit(start_date),
                F.lit(end_date),
                F.expr("INTERVAL 1 DAY"),
            )
        ).alias("calendar_date")
    )

    return calendar_df.select(
        F.date_format("calendar_date", "yyyyMMdd")
        .cast("int")
        .alias("date_key"),
        F.col("calendar_date"),
        F.year("calendar_date").alias("year"),
        F.quarter("calendar_date").alias("quarter"),
        F.month("calendar_date").alias("month_number"),
        F.date_format("calendar_date", "MMMM").alias("month_name"),
        F.date_format("calendar_date", "yyyy-MM").alias("year_month"),
    )


def prepare_unit_dimension(silver_source_df):
    """Prepare supported measurement units with stable keys."""
    units_df = silver_source_df.select(
        F.col("units").alias("unit_code")
    ).distinct()

    invalid_units = units_df.filter(
        F.col("unit_code").isNull()
        | ~F.col("unit_code").isin(["MBBL", "MBBL/D"])
    )

    if invalid_units.limit(1).count() > 0:
        raise ValueError(
            "Silver contains missing or unsupported measurement units."
        )

    return (
        units_df
        .withColumn(
            "unit_key",
            F.sha2(
                F.to_json(
                    F.struct(
                        F.lit("unit").alias("dimension"),
                        F.col("unit_code").alias("code"),
                    )
                ),
                256,
            ),
        )
        .withColumn(
            "unit_description",
            F.when(
                F.col("unit_code") == "MBBL",
                F.lit("Thousand barrels"),
            ).otherwise(F.lit("Thousand barrels per day")),
        )
        .select("unit_key", "unit_code", "unit_description")
    )


print("Gold dimension preparation functions ready.")

In [0]:
def prepare_gold_fact(silver_source_df):
    """Resolve dimension keys and prepare validated fact records."""
    source_count = silver_source_df.count()

    if source_count == 0:
        raise ValueError("Cannot prepare facts from empty Silver data.")

    invalid_keys = (
        silver_source_df.groupBy("period", "series")
        .count()
        .filter(
            F.col("period").isNull()
            | F.col("series").isNull()
            | (F.trim(F.col("series")) == "")
            | (F.col("count") > 1)
        )
    )

    if invalid_keys.limit(1).count() > 0:
        raise ValueError("Silver contains missing or duplicate business keys.")

    fact_df = silver_source_df.select(
        "period",
        "series",
        "product",
        "duoarea",
        "process",
        "units",
        "value",
        F.col("load_timestamp").alias("silver_load_timestamp"),
    )

    date_lookup = spark.table(gold_tables["date"]).select(
        F.col("calendar_date").alias("period"),
        "date_key",
    )

    fact_df = fact_df.join(date_lookup, on="period", how="left")

    dimension_joins = [
        ("product", "product", "product_code", "product_key"),
        ("area", "duoarea", "area_code", "area_key"),
        ("process", "process", "process_code", "process_key"),
        ("unit", "units", "unit_code", "unit_key"),
    ]

    for role, source_code, dimension_code, key_column in dimension_joins:
        lookup_df = spark.table(gold_tables[role]).select(
            F.col(dimension_code).alias(source_code),
            key_column,
        )

        fact_df = fact_df.join(lookup_df, on=source_code, how="left")

    foreign_keys = [
        "date_key",
        "product_key",
        "area_key",
        "process_key",
        "unit_key",
    ]

    missing_match = F.lit(False)
    for key_column in foreign_keys:
        missing_match = missing_match | F.col(key_column).isNull()

    if fact_df.filter(missing_match).limit(1).count() > 0:
        raise ValueError("Some observations have no matching dimension member.")

    if fact_df.count() != source_count:
        raise ValueError("Dimension joins changed the observation count.")

    fact_df = fact_df.withColumn(
        "observation_key",
        F.sha2(F.to_json(F.struct("period", "series")), 256),
    )

    business_columns = foreign_keys + ["series", "value"]

    fact_df = fact_df.withColumn(
        "record_hash",
        F.sha2(
            F.to_json(
                F.struct(*[F.col(name) for name in business_columns]),
                options={"ignoreNullFields": "false"},
            ),
            256,
        ),
    )

    return fact_df.select(
        "observation_key",
        *business_columns,
        "record_hash",
        "silver_load_timestamp",
    )


def write_gold_execution_log(
    *,
    run_id,
    batch_id,
    load_type,
    source_reference,
    target_table,
    start_time,
    end_time,
    status,
    rows_read=None,
    rows_output=None,
    rows_aggregated=None,
    rows_inserted=None,
    rows_updated=None,
    error_message=None,
):
    """Append one Silver-to-Gold operation to the shared audit table."""
    if status not in {"Success", "Failure"}:
        raise ValueError("Status must be Success or Failure.")

    log_schema = spark.table(execution_log_table).schema
    log_record = {field.name: None for field in log_schema.fields}

    log_record.update({
        "log_id": str(uuid4()),
        "run_id": run_id,
        "batch_id": batch_id,
        "load_type": load_type,
        "layer": "Silver-to-Gold",
        "source_reference": str(source_reference),
        "target_table": target_table,
        "start_time": start_time,
        "end_time": end_time,
        "status": status,
        "rows_read": rows_read,
        "rows_output": rows_output,
        "rows_aggregated": rows_aggregated,
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
        "rows_quarantined": 0,
        "error_message": error_message,
        "load_timestamp": datetime.now(timezone.utc),
    })

    (
        spark.createDataFrame([log_record], schema=log_schema)
        .write.format("delta")
        .mode("append")
        .saveAsTable(execution_log_table)
    )


print("Gold fact preparation and logging functions ready.")

In [0]:
def _load_gold_table(
    silver_source_df,
    *,
    table_role,
    batch_id,
    load_type,
    source_reference,
):
    """Prepare and merge one Gold table with execution auditing."""
    run_id = str(uuid4())
    start_time = datetime.now(timezone.utc)
    target_table = gold_tables[table_role]
    view_name = f"gold_source_{uuid4().hex}"

    rows_read = None
    rows_output = None
    rows_inserted = None
    rows_updated = None
    status = "Failure"
    error_message = None

    try:
        if not isinstance(batch_id, str) or not batch_id.strip():
            raise ValueError("Batch ID must be a nonempty string.")

        if load_type not in {"full_load", "incremental_load"}:
            raise ValueError("Unsupported load type.")

        rows_read = silver_source_df.count()

        if rows_read == 0:
            raise ValueError("Cannot load Gold from empty Silver data.")

        if table_role == "fact":
            prepared_df = prepare_gold_fact(silver_source_df)
            key_column = "observation_key"
            changed_condition = (
                "NOT (target.record_hash <=> source.record_hash)"
            )

        else:
            if table_role == "date":
                prepared_df = prepare_date_dimension(silver_source_df)
            elif table_role == "unit":
                prepared_df = prepare_unit_dimension(silver_source_df)
            else:
                prepared_df = prepare_descriptive_dimension(
                    silver_source_df,
                    dimension_role=table_role,
                )

            key_column = f"{table_role}_key"

            attribute_columns = [
                name for name in prepared_df.columns
                if name != key_column
            ]

            changed_condition = " OR ".join(
                f"NOT (target.`{name}` <=> source.`{name}`)"
                for name in attribute_columns
            )

        rows_output = prepared_df.count()

        invalid_keys = (
            prepared_df.groupBy(key_column)
            .count()
            .filter(
                F.col(key_column).isNull()
                | (F.col("count") > 1)
            )
        )

        if invalid_keys.limit(1).count() > 0:
            raise ValueError("Prepared Gold data has null or duplicate keys.")

        source_df = prepared_df.withColumn(
            "load_timestamp",
            F.lit(start_time).cast("timestamp"),
        )

        if table_role == "fact":
            source_df = source_df.withColumn("run_id", F.lit(run_id))

        source_df = source_df.select(
            *gold_schemas[table_role].fieldNames()
        )

        source_df.createOrReplaceTempView(view_name)

        merge_result = spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING {view_name} AS source
            ON target.`{key_column}` = source.`{key_column}`

            WHEN MATCHED AND ({changed_condition})
            THEN UPDATE SET *

            WHEN NOT MATCHED
            THEN INSERT *
        """).collect()

        if len(merge_result) != 1:
            raise RuntimeError("MERGE did not return operation metrics.")

        metrics = merge_result[0].asDict()

        if not {"num_inserted_rows", "num_updated_rows"}.issubset(metrics):
            raise RuntimeError("Required MERGE metrics are unavailable.")

        rows_inserted = int(metrics["num_inserted_rows"])
        rows_updated = int(metrics["num_updated_rows"])
        status = "Success"

    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        raise

    finally:
        end_time = datetime.now(timezone.utc)

        try:
            write_gold_execution_log(
                run_id=run_id,
                batch_id=batch_id,
                load_type=load_type,
                source_reference=source_reference,
                target_table=target_table,
                start_time=start_time,
                end_time=end_time,
                status=status,
                rows_read=rows_read,
                rows_output=rows_output,
                # No petroleum measure aggregation occurs here.
                rows_aggregated=0,
                rows_inserted=rows_inserted,
                rows_updated=rows_updated,
                error_message=error_message,
            )
        finally:
            spark.catalog.dropTempView(view_name)

    return {
        "run_id": run_id,
        "status": status,
        "rows_read": rows_read,
        "rows_output": rows_output,
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
    }


def load_gold_dimension(
    silver_source_df,
    *,
    dimension_role,
    batch_id,
    load_type,
    source_reference=None,
):
    """Load one dimension using the shared audited MERGE helper."""
    if dimension_role not in {"date", "product", "area", "process", "unit"}:
        raise ValueError(f"Unsupported dimension: {dimension_role}")

    result = _load_gold_table(
        silver_source_df,
        table_role=dimension_role,
        batch_id=batch_id,
        load_type=load_type,
        source_reference=source_reference or (
            f"{silver_table}; dimension={dimension_role}; "
            "scope=provided Silver DataFrame"
        ),
    )

    result["dimension"] = dimension_role
    return result


def load_gold_fact(
    silver_source_df,
    *,
    batch_id,
    load_type,
    source_reference=None,
):
    """Load observation facts using the shared audited MERGE helper."""
    return _load_gold_table(
        silver_source_df,
        table_role="fact",
        batch_id=batch_id,
        load_type=load_type,
        source_reference=source_reference or (
            f"{silver_table}; scope=provided current Silver observations"
        ),
    )


def refresh_gold(*, batch_id, load_type):
    """Refresh all Gold tables from one consistent Silver snapshot."""
    if not isinstance(batch_id, str) or not batch_id.strip():
        raise ValueError("Batch ID must be a nonempty string.")

    if load_type not in {"full_load", "incremental_load"}:
        raise ValueError("Unsupported load type.")

    silver_version = (
        spark.sql(f"DESCRIBE HISTORY {silver_table}")
        .orderBy(F.col("version").desc())
        .select("version")
        .first()["version"]
    )

    source_df = (
        spark.read
        .option("versionAsOf", str(silver_version))
        .table(silver_table)
    )

    source_reference = (
        f"{silver_table}; version={silver_version}; "
        "scope=complete current Silver snapshot"
    )

    print(f"Refreshing Gold from Silver version: {silver_version}")

    dimension_results = []

    for dimension_role in ["date", "product", "area", "process", "unit"]:
        result = load_gold_dimension(
            source_df,
            dimension_role=dimension_role,
            batch_id=batch_id,
            load_type=load_type,
            source_reference=f"{source_reference}; dimension={dimension_role}",
        )
        dimension_results.append(result)

        print(
            f"{dimension_role}: "
            f"inserted={result['rows_inserted']:,}, "
            f"updated={result['rows_updated']:,}"
        )

    fact_result = load_gold_fact(
        source_df,
        batch_id=batch_id,
        load_type=load_type,
        source_reference=source_reference,
    )

    print(
        f"fact: inserted={fact_result['rows_inserted']:,}, "
        f"updated={fact_result['rows_updated']:,}"
    )

    # Aggregate runs only after the fact table is up to date.
    agg_result = load_gold_aggregate(
        batch_id=batch_id,
        load_type=load_type,
    )

    print(
        f"agg: inserted={agg_result['rows_inserted']:,}, "
        f"updated={agg_result['rows_updated']:,}"
    )

    return {
        "silver_version": silver_version,
        "dimension_results": dimension_results,
        "fact_result": fact_result,
        "agg_result": agg_result,
    }


print("Audited Gold loaders ready.")
print("Gold refresh orchestrator ready.")

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType,
)

# Register the aggregate table alongside the other Gold tables.
gold_tables["agg_monthly"] = f"{gold_table_prefix}_agg_monthly_product_process"

agg_schema = StructType([
    StructField("agg_key", StringType(), False),
    StructField("date_key", IntegerType(), False),
    StructField("product_key", StringType(), False),
    StructField("process_key", StringType(), False),
    StructField("unit_key", StringType(), False),
    StructField("padd_total_value", DoubleType(), True),
    StructField("series_count", IntegerType(), False),
    StructField("load_timestamp", TimestampType(), False),
])

agg_columns_sql = ",\n".join(
    f"`{field.name}` {field.dataType.simpleString()}"
    + ("" if field.nullable else " NOT NULL")
    for field in agg_schema.fields
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_tables['agg_monthly']} (
        {agg_columns_sql}
    )
    USING DELTA
""")


def load_gold_aggregate(*, batch_id, load_type):
    """Sum the five PADD regions per month, product, process, and unit.

    U.S. total area codes are excluded so regional values are not
    double counted. MBBL and MBBL/D remain separate via unit_key.
    """
    run_id = str(uuid4())
    start_time = datetime.now(timezone.utc)
    target_table = gold_tables["agg_monthly"]
    view_name = f"gold_agg_{uuid4().hex}"

    rows_read = None
    rows_output = None
    rows_inserted = None
    rows_updated = None
    status = "Failure"
    error_message = None

    try:
        if not isinstance(batch_id, str) or not batch_id.strip():
            raise ValueError("Batch ID must be a nonempty string.")

        if load_type not in {"full_load", "incremental_load"}:
            raise ValueError("Unsupported load type.")

        fact_df = spark.table(gold_tables["fact"])
        area_df = spark.table(gold_tables["area"])

        rows_read = fact_df.count()

        # Regional PADD codes only (R10..R50, with or without -Z00).
        regional_keys_df = (
            area_df
            .filter(F.col("area_code").startswith("R"))
            .select("area_key")
        )

        regional_df = fact_df.join(regional_keys_df, "area_key", "inner")

        agg_df = (
            regional_df
            .groupBy("date_key", "product_key", "process_key", "unit_key")
            .agg(
                F.sum("value").alias("padd_total_value"),
                F.count("*").cast("int").alias("series_count"),
            )
            .withColumn(
                "agg_key",
                F.sha2(
                    F.to_json(
                        F.struct(
                            "date_key",
                            "product_key",
                            "process_key",
                            "unit_key",
                        )
                    ),
                    256,
                ),
            )
            .withColumn(
                "load_timestamp",
                F.lit(start_time).cast("timestamp"),
            )
            .select(*agg_schema.fieldNames())
        )

        rows_output = agg_df.count()

        agg_df.createOrReplaceTempView(view_name)

        merge_result = spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING {view_name} AS source
            ON target.agg_key = source.agg_key

            WHEN MATCHED AND (
                NOT (target.padd_total_value <=> source.padd_total_value)
                OR NOT (target.series_count <=> source.series_count)
            )
            THEN UPDATE SET *

            WHEN NOT MATCHED
            THEN INSERT *
        """).collect()

        if len(merge_result) != 1:
            raise RuntimeError("MERGE did not return operation metrics.")

        metrics = merge_result[0].asDict()

        if not {"num_inserted_rows", "num_updated_rows"}.issubset(metrics):
            raise RuntimeError("Required MERGE metrics are unavailable.")

        rows_inserted = int(metrics["num_inserted_rows"])
        rows_updated = int(metrics["num_updated_rows"])
        status = "Success"

    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        raise

    finally:
        end_time = datetime.now(timezone.utc)

        try:
            write_gold_execution_log(
                run_id=run_id,
                batch_id=batch_id,
                load_type=load_type,
                source_reference=(
                    f"{gold_tables['fact']}; "
                    "regional PADD monthly aggregation"
                ),
                target_table=target_table,
                start_time=start_time,
                end_time=end_time,
                status=status,
                rows_read=rows_read,
                rows_output=rows_output,
                # Fact rows read and combined into aggregate rows.
                rows_aggregated=rows_read,
                rows_inserted=rows_inserted,
                rows_updated=rows_updated,
                error_message=error_message,
            )
        finally:
            spark.catalog.dropTempView(view_name)

    return {
        "run_id": run_id,
        "status": status,
        "rows_read": rows_read,
        "rows_output": rows_output,
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
    }


print("Gold aggregate loader ready.")